In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 加载model,tokenizer
custom_model_name = "fine-tuned-models/nllb-200-distilled-600M/zh2ko_1101"

model = AutoModelForSeq2SeqLM.from_pretrained(custom_model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(custom_model_name, device_map="auto")

In [ ]:
import pandas as pd

df = pd.read_excel("all_files_merged_zh-CN_ko_valid_tagged.xlsx")

source = df["zh-CN"].to_list()  # 待翻译的句子
references = df["ko"].to_list()  # 标准的翻译

In [ ]:
from tqdm.notebook import tqdm

inputs = [tokenizer(item, return_tensors="pt").to('cuda') for item in tqdm(source)]
translated_tokens = [
    model.generate(**input, forced_bos_token_id=tokenizer.lang_code_to_id[tokenizer.tgt_lang], max_length=400)
    for input in tqdm(inputs)]
translated_text = [tokenizer.batch_decode(translated_token, skip_special_tokens=False) for translated_token in
                   translated_tokens]

In [ ]:
df_compare = pd.DataFrame({
    "source": source,
    "references": references,
    "candidates": translated_text,
})
df_compare

In [ ]:
file_name = "translation_result_of_{0}".format(custom_model_name.replace("/", "_"))
df_compare.to_excel("{0}.xlsx".format(file_name), index=False)
df_compare.to_csv("{0}.csv".format(file_name), index=False)